In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from sklearn.svm import SVC

In [25]:
df = pd.read_csv("titanic_data_updated.csv")

# Display 5 random rows to see what the data looks like
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
630,631,yes,first,"Barkworth, Mr. Algernon Henry Wilson",male,80.0,0,0,27042,30.0000,A23,S
768,769,no,third,"Moran, Mr. Daniel J",male,NaN,1,0,371110,24.1500,NaN,Q
437,438,yes,second,"Richards, Mrs. Sidney (Emily Hocking)",female,24.0,2,3,29106,18.7500,NaN,S
605,606,no,third,"Lindell, Mr. Edvard Bengtsson",male,36.0,1,0,349910,15.5500,NaN,S
692,693,yes,third,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,NaN,S


In [26]:
df['Cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


In [27]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
309,310,yes,first,"Francatelli, Miss. Laura Mabel",female,30.0,0,0,PC 17485,56.9292,E36,C,1,E
301,302,yes,third,"McCoy, Mr. Bernard",male,NaN,2,0,367226,23.2500,Missing,Q,3,M
359,360,yes,third,"Mockler, Miss. Helen Mary ""Ellie""",female,NaN,0,0,330980,7.8792,Missing,Q,1,M
370,371,yes,first,"Harder, Mr. George Achilles",male,25.0,1,0,11765,55.4417,E50,C,2,E
518,519,yes,second,"Angle, Mrs. William A (Florence ""Mary"" Agnes H...",female,36.0,1,0,226875,26.0000,Missing,S,2,M


In [28]:
df['Deck'].value_counts()

Deck
M    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [29]:
# Separate the features (X) from the target we want to predict (y)
# We drop 'Survived' from X because that's our answer key
X = df.drop('Survived', axis=1)

# y contains only the survival status
y = df['Survived']

In [30]:
# Split data: 80% for training the model and 20% for testing its accuracy
# 'stratify=y' ensures both sets have a similar percentage of survivors
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [31]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
692,693,third,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,Missing,S,1,M
481,482,second,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,Missing,S,1,M
527,528,first,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,1,C
855,856,third,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,Missing,S,2,M
801,802,second,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,Missing,S,3,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,360,third,"Mockler, Miss. Helen Mary ""Ellie""",female,NaN,0,0,330980,7.8792,Missing,Q,1,M
258,259,first,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,Missing,C,1,M
736,737,third,"Ford, Mrs. Edward (Margaret Ann Watson)",female,48.0,1,3,W./C. 6608,34.3750,Missing,S,5,M
462,463,first,"Gee, Mr. Arthur H",male,47.0,0,0,111320,38.5000,E63,S,1,E


In [32]:
#age
mean_age = X_train['Age'].mean()
std_age = X_train['Age'].std()

X_train['Z_score'] = (X_train['Age'] - mean_age) / std_age

musk = (abs(X_train['Z_score']) <= 3)

X_train = X_train[musk]
y_train = y_train[musk]

# fare

fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)

IQR = fare_Q3 - fare_Q1

minimum = max(0 , fare_Q1 - 1.5 * IQR)
maximum = fare_Q3 + 1.5 * IQR

X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)

C:\Users\user\AppData\Local\Temp\ipykernel_11132\987313337.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)


In [33]:
# pipeline

# numerical
p1 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='mean')),
        ('scaler',StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',MinMaxScaler())
    ]
)

In [34]:
# categories goes one after another via stricly following the order of the input
categories = [['third','second','first']]

In [35]:
# pipeline

#  categorical columns

p3 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore'))
    ]
)

p4 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OrdinalEncoder(categories=categories)),
        ('scaler',MinMaxScaler())
    ]
)

In [36]:
preprocessor = ColumnTransformer(
    transformers=[
        ('pipeline_1',p1,['Age']),
        ('pipeline_2',p2,['Fare','Family_Size']),
        ('pipeline_3',p3,['Embarked','Sex','Deck']),
        ('pipeline_4',p4,['Pclass'])
    ],
    remainder='drop'
)
preprocessor

,transformers,"[('pipeline_1', ...), ('pipeline_2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


In [37]:
le = LabelEncoder()

le.fit(y_train)

y_train = le.transform(y_train)
y_test = le.transform(y_test)


# model Training

In [38]:
SVC_model=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',SVC())
    ]
)

In [39]:
from sklearn.model_selection import GridSearchCV

In [59]:
grid_param = [
    {
        "kernel": ['linear'],
        "C": [0.01, 0.1, 1, 10, 50, 100]
    },
    {
        "kernel": ['rbf'],
        "C": [0.01, 0.1, 1, 100],
        "gamma": [0.01, 0.1, 5, 10, 'scale', 'auto']
    },
    {
        "kernel": ['poly'],
        "C": [0.01, 0.1, 1, 100],
        "degree": [2, 3]
    }
]

In [64]:
best_SVC_model = GridSearchCV(estimator = SVC_model , param_grid = grid_param , cv = 5)

In [65]:
best_SVC_model.fit(X_train,y_train)

C:\Users\user\AppData\Roaming\Python\Python312\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\user\AppData\Roamin

,estimator,"Pipeline(step...del', SVC())])"
,param_grid,"[{'model__C': [0.01, 0.1, ...], 'model__kernel': ['linear']}, {'model__C': [0.01, 0.1, ...], 'model__gamma': [0.01, 0.1, ...], 'model__kernel': ['rbf']}, ...]"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('pipeline_1', ...), ('pipeline_2', ...), ...]"


In [66]:

from sklearn.metrics import accuracy_score


y_pred_train = best_SVC_model.predict(X_train)
print("Training Accuracy:", accuracy_score(y_train,y_pred_train))

Training Accuracy: 0.8429319371727748
